In [1]:
import os
import numpy as np
import rasterio as rio
from shapely.geometry import Point
import geopandas as gpd
from tqdm import tqdm

# -------------------------------
# 1. 설정
# -------------------------------
base_dir = "./Data"
utm_epsg_uzb = 32641
boundary_shp_path = "./Boundary_lines/uzb_grid_1km/uzb_boundary_grid_1km.shp"
grid_gdf = gpd.read_file(boundary_shp_path).to_crs(epsg=utm_epsg_uzb)

# -------------------------------
# 2. 반복 처리할 변수 정의
# -------------------------------
years = list(range(2000, 2021))
genders = ["f", "m"]
age_groups = [0, 1, 5] + list(range(10, 85, 5))  # 0, 1, 5, 10, 15, ..., 80

# -------------------------------
# 3. 반복 처리
# -------------------------------
for year in years:
    print(f"\n📅 {year}년 데이터 처리 시작")
    year_dir = os.path.join(base_dir, str(year))
    input_dir = os.path.join(year_dir, "input")
    point_dir = os.path.join(year_dir, "point")
    output_dir = os.path.join(year_dir, "output")
    
    os.makedirs(input_dir, exist_ok=True)
    os.makedirs(point_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)

    for gender in genders:
        for age in age_groups:
            file_stem = f"uzb_{gender}_{age}_{year}_nukus" # 나중에 전체 지역으로 처리할때는 _nukus 제거하자
            input_raster = os.path.join(input_dir, f"{file_stem}.tif")
            output_point = os.path.join(point_dir, f"{file_stem}.geojson")
            output_grid = os.path.join(output_dir, f"{file_stem}_1km_joined.geojson")

            if not os.path.exists(input_raster):
                print(f"❌ 없음: {input_raster}")
                continue

            # -------------------------------------
            # 1단계: 레스터 → 포인트 GeoJSON
            # -------------------------------------
            with rio.open(input_raster) as src:
                data = src.read(1)
                transform = src.transform
                nodata = src.nodata

                rows, cols = np.where(data != nodata)
                points = [Point(transform * (col + 0.5, row + 0.5)) for row, col in zip(rows, cols)]
                values = [data[row, col] for row, col in zip(rows, cols)]

                gdf = gpd.GeoDataFrame({'geometry': points, 'population': values}, crs=src.crs)
                gdf = gdf.to_crs(epsg=utm_epsg_uzb)
                gdf.to_file(output_point, driver="GeoJSON")

            # -------------------------------------
            # 2단계: 격자와 공간 조인 및 집계
            # -------------------------------------
            point_gdf = gpd.read_file(output_point).to_crs(epsg=utm_epsg_uzb)
            joined_gdf = gpd.sjoin(point_gdf, grid_gdf, how="left", predicate="intersects")

            # 격자별 인구 집계
            grid_population = joined_gdf.groupby("index_right")["population"].sum().reset_index()

            # 격자에 집계 정보 병합
            grid_gdf["grid_id"] = grid_gdf.index
            result = grid_gdf.merge(grid_population, left_on="grid_id", right_on="index_right", how="left")

            # ✅ 결측값 처리 포함 
            result["population"] = result["population"].fillna(0).astype(int)

            # 결과 저장
            result.to_file(f"{output_grid}, driver="GeoJSON")
            print(f"✅ 저장 완료: {output_grid}")



📅 2000년 데이터 처리 시작
❌ 없음: ./Data\2000\input\uzb_f_0_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_f_1_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_f_5_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_f_10_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_f_15_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_f_20_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_f_25_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_f_30_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_f_35_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_f_40_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_f_45_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_f_50_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_f_55_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_f_60_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_f_65_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_f_70_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_f_75_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_f_80_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_m_0_2000_nukus.tif
❌ 없음: ./Data\2000\input\uzb_m_1_2000_nukus.tif
❌ 없음: ./Data\2000\input\uz